<a href="https://colab.research.google.com/github/jostina/Netflix-Dataset/blob/main/Task_3%E2%80%93_Netflix_Content_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Import the libraries and load the dataset

In [ ]:
# Step 1: Import the required libraries

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
%pip install pandas numpy scikit-learn

Step 2: Load the cleaned Netflix dataset

In [ ]:
# Step 2: Load the dataset

df = pd.read_csv("/content/netflix_titles_cleaned.csv")

# Display the first five rows
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,date_added_year,date_added_month,date_added_month_name,duration_value,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,9,September,90,min
1,s3,Tv Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,9,September,1,Season
2,s6,Tv Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,9,September,1,Season
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,9,September,91,min
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,9,September,125,min


In [ ]:
# Number of rows and columns
print("Dataset shape:", df.shape)

# Column names
print("Column names:", df.columns.tolist())

# Check missing values
print(df.isnull().sum())

Dataset shape: (8790, 15)
Column names: ['show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'date_added_year', 'date_added_month', 'date_added_month_name', 'duration_value', 'duration_unit']
show_id                  0
type                     0
title                    0
director                 0
country                  0
date_added               0
release_year             0
rating                   0
duration                 0
listed_in                0
date_added_year          0
date_added_month         0
date_added_month_name    0
duration_value           0
duration_unit            0
dtype: int64


Step 3: Select the features
Features are the pieces of information the model uses to compare Netflix titles.

For this project, we will use:
- type: Movie or TV Show.
- listed_in: Genres and categories.
- director: The director of the title.
- country: The country of production.
- rating: The content rating.

In [ ]:
# Step 3: Select relevant features

features = [
    "type",
    "listed_in",
    "director",
    "country",
    "rating"
]

# Create a separate DataFrame containing the selected features
content_df = df[["title"] + features].copy()

# Display the selected columns
content_df.head()

,title,type,listed_in,director,country,rating
0,Dick Johnson Is Dead,Movie,Documentaries,Kirsten Johnson,United States,PG-13
1,Ganglands,Tv Show,"Crime TV Shows, International TV Shows, TV Act...",Julien Leclercq,France,TV-MA
2,Midnight Mass,Tv Show,"TV Dramas, TV Horror, TV Mysteries",Mike Flanagan,United States,TV-MA
3,Confessions of an Invisible Girl,Movie,"Children & Family Movies, Comedies",Bruno Garotti,Brazil,TV-PG
4,Sankofa,Movie,"Dramas, Independent Movies, International Movies",Haile Gerima,United States,TV-MA


Step 4: Handle missing values

In [ ]:
# Step 4: Replace missing values with empty strings

for column in features:
    content_df[column] = content_df[column].fillna("")

# Check for missing values
print(content_df[features].isnull().sum())

type         0
listed_in    0
director     0
country      0
rating       0
dtype: int64


Step 5: Combine the features into one column

In [ ]:
# Step 5: Combine the features

content_df["content"] = (
    content_df["type"] + " " +
    content_df["listed_in"] + " " +
    content_df["director"] + " " +
    content_df["country"] + " " +
    content_df["rating"]
)

# Display the title and combined content
content_df[["title", "content"]].head()

,title,content
0,Dick Johnson Is Dead,Movie Documentaries Kirsten Johnson United Sta...
1,Ganglands,"Tv Show Crime TV Shows, International TV Shows..."
2,Midnight Mass,"Tv Show TV Dramas, TV Horror, TV Mysteries Mik..."
3,Confessions of an Invisible Girl,"Movie Children & Family Movies, Comedies Bruno..."
4,Sankofa,"Movie Dramas, Independent Movies, Internationa..."


Step 6: Convert text into numerical features using TF-IDF
Computers need numerical representations to calculate similarity between text records.

We will use TF-IDF (Term Frequency–Inverse Document Frequency) to convert each title's combined content into a numerical vector.

In [ ]:
# Step 6: Initialize the TF-IDF vectorizer

tfidf = TfidfVectorizer(
    stop_words="english"
)

# Convert the content column into numerical vectors
tfidf_matrix = tfidf.fit_transform(
    content_df["content"]
)

# Display the dimensions of the resulting matrix
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (8790, 6569)


Step 7: Calculate similarity between titles

In [ ]:
# Step 7: Calculate cosine similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

# Check the shape
print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (8790, 8790)


Step 8: Create the recommendation function
This is the main part of the project.

The function will:
1. Accept a Netflix title.
2. Find that title in the dataset.
3. Retrieve its similarity scores.
4. Rank other titles by similarity.
5. Return the five most similar titles.

In [ ]:
# Step 8: Create the recommendation function

def recommend_titles(title, n=5):

    # Find the selected title
    title_index = content_df[
        content_df["title"].str.casefold() == title.casefold()
    ].index

    # Check whether the title exists
    if len(title_index) == 0:
        print("Title not found. Please check the spelling.")
        return

    # Get the index of the selected title
    index = title_index[0]

    # Get similarity scores for this title
    scores = list(enumerate(similarity_matrix[index]))

    # Sort titles from highest to lowest similarity
    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the selected title itself
    scores = [
        item for item in scores
        if item[0] != index
    ]

    # Select the top n recommendations
    scores = scores[:n]

    # Get the recommended title indices
    recommended_indices = [
        item[0] for item in scores
    ]

    # Retrieve the recommended titles
    recommendations = content_df.iloc[
        recommended_indices
    ][["title", "type", "listed_in"]].copy()

    # Add similarity scores
    recommendations["similarity_score"] = [
        round(item[1], 3) for item in scores
    ]

    return recommendations.reset_index(drop=True)


In [ ]:
# Let's test the recommendation function instead!
recommend_titles("Midnight Mass")

,title,type,listed_in,similarity_score
0,Gerald's Game,Movie,"Horror Movies, Thrillers",0.802
1,Hush,Movie,"Horror Movies, Thrillers",0.753
2,Before I Wake,Movie,"Horror Movies, Thrillers",0.703
3,Brand New Cherry Flavor,Tv Show,"TV Dramas, TV Horror, TV Mysteries",0.670
4,The Haunting of Bly Manor,Tv Show,"TV Dramas, TV Horror, TV Mysteries",0.670


Step 9: Generate recommendations

In [ ]:
# Recommend five titles similar to Midnight Mass

recommend_titles("Midnight Mass", n=5)

,title,type,listed_in,similarity_score
0,Gerald's Game,Movie,"Horror Movies, Thrillers",0.802
1,Hush,Movie,"Horror Movies, Thrillers",0.753
2,Before I Wake,Movie,"Horror Movies, Thrillers",0.703
3,Brand New Cherry Flavor,Tv Show,"TV Dramas, TV Horror, TV Mysteries",0.670
4,The Haunting of Bly Manor,Tv Show,"TV Dramas, TV Horror, TV Mysteries",0.670


In [ ]:
recommend_titles("Stranger Things", n=5)

,title,type,listed_in,similarity_score
0,Chilling Adventures of Sabrina,Tv Show,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy",1.000
1,Nightflyers,Tv Show,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy",0.968
2,Helix,Tv Show,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy",0.968
3,Manifest,Tv Show,"TV Dramas, TV Mysteries, TV Sci-Fi & Fantasy",0.934
4,The Vampire Diaries,Tv Show,"TV Dramas, TV Mysteries, TV Sci-Fi & Fantasy",0.934


In [ ]:
recommend_titles("The Crown", n=5)

,title,type,listed_in,similarity_score
0,Behind Her Eyes,Tv Show,"British TV Shows, International TV Shows, TV D...",1.0
1,Black Mirror,Tv Show,"British TV Shows, International TV Shows, TV D...",1.0
2,Traitors,Tv Show,"British TV Shows, International TV Shows, TV D...",1.0
3,Black Earth Rising,Tv Show,"British TV Shows, International TV Shows, TV D...",1.0
4,Requiem,Tv Show,"British TV Shows, International TV Shows, TV D...",1.0


In [ ]:
recommend_titles("Breaking Bad", n=5)

,title,type,listed_in,similarity_score
0,The Assassination of Gianni Versace,Tv Show,"Crime TV Shows, TV Dramas, TV Thrillers",1.000
1,Dare Me,Tv Show,"Crime TV Shows, TV Dramas, TV Thrillers",1.000
2,Ozark,Tv Show,"Crime TV Shows, TV Dramas, TV Thrillers",1.000
3,Designated Survivor,Tv Show,"Crime TV Shows, TV Dramas, TV Thrillers",1.000
4,The Blacklist,Tv Show,"Crime TV Shows, TV Dramas, TV Thrillers",0.947


Step 10: Evaluate the recommendation system
For this project, we'll evaluate whether the recommended titles share categories with the selected title.

We will calculate a simple metric called Category-Hit Rate@5.

The metric measures how many of the five recommended titles share at least one category with the original title.
Example
Suppose the original title belongs to these categories:TV Dramas, TV Horror.

In [ ]:
# Step 10: Evaluate the recommendations

def evaluate_recommendations(title, n=5):

    # Get the original title's categories
    original = content_df[
        content_df["title"].str.casefold() == title.casefold()
    ]

    if original.empty:
        print("Title not found.")
        return

    original_categories = set(
        original.iloc[0]["listed_in"].split(", ")
    )

    # Generate recommendations
    recommendations = recommend_titles(title, n)

    if recommendations is None:
        return

    # Count recommendations sharing at least one category
    hits = 0

    for categories in recommendations["listed_in"]:

        recommended_categories = set(
            categories.split(", ")
        )

        if original_categories & recommended_categories:
            hits += 1

    # Calculate category-hit rate
    hit_rate = hits / len(recommendations)

    print("Original title:", title)
    print("Number of recommendations:", len(recommendations))
    print("Category hits:", hits)
    print(f"Category-Hit Rate: {hit_rate:.2%}")

    return recommendations

In [ ]:
evaluate_recommendations("Midnight Mass", n=5)

Original title: Midnight Mass
Number of recommendations: 5
Category hits: 2
Category-Hit Rate: 40.00%


,title,type,listed_in,similarity_score
0,Gerald's Game,Movie,"Horror Movies, Thrillers",0.802
1,Hush,Movie,"Horror Movies, Thrillers",0.753
2,Before I Wake,Movie,"Horror Movies, Thrillers",0.703
3,Brand New Cherry Flavor,Tv Show,"TV Dramas, TV Horror, TV Mysteries",0.670
4,The Haunting of Bly Manor,Tv Show,"TV Dramas, TV Horror, TV Mysteries",0.670


### Conclusion

The project developed a content-based recommendation system using the cleaned Netflix titles dataset. The system uses Pandas for data preparation, TF-IDF for text feature extraction, and cosine similarity to identify similar movies and TV shows.

Relevant metadata, including content type, categories, director, country, and rating, was combined to represent each title. A recommendation function was then developed to retrieve the most similar titles for a selected movie or TV show.

Recommendation quality was assessed using category overlap between the selected title and its recommendations. This provided a basic measure of how consistently the system recommended titles with related categories.

The model demonstrates how machine learning techniques can support content discovery without requiring user ratings or viewing histories. However, its effectiveness is limited by the available metadata. Future improvements could include plot descriptions, cast information, popularity data, and user interaction histories to produce more informative recommendations.